# `partial_json_parser` smoke test — parses JSON that is still being streamed

vLLM's tool-call parsers use it to read a tool call's arguments while the model is still writing them. Pure Python, no network, no GPU.

## Example 1 — the installed version, and a tool call cut off mid-stream

In [ ]:
import importlib.metadata
import partial_json_parser
from partial_json_parser.core.options import Allow
from partial_json_parser.version import __version__ as module_version

version = importlib.metadata.version("partial_json_parser")
assert version and version == module_version, (version, module_version)
cut_off = '{"name": "get_weather", "arguments": {"city": "Par'
parsed = partial_json_parser.loads(cut_off, Allow.ALL)
assert parsed == {"name": "get_weather", "arguments": {"city": "Par"}}, parsed
print("partial_json_parser", version, "->", parsed)

## Example 2 — `Allow` flags decide which unfinished values are kept

In [ ]:
from partial_json_parser import loads
from partial_json_parser.core.options import Allow

cut_off = '{"name": "get_weather", "arguments": {"city": "Par'
# vLLM's parsers pass `Allow.ALL & ~Allow.STR` so a half-written string is dropped instead of shown.
without_strings = loads(cut_off, Allow.ALL & ~Allow.STR)
assert without_strings == {"name": "get_weather", "arguments": {}}, without_strings
assert loads('[1, 2, 3.', Allow.ARR) == [1, 2], loads('[1, 2, 3.', Allow.ARR)
assert loads('[1, 2, 3.', Allow.ARR | Allow.NUM) == [1, 2, 3.0]
print("string dropped:", without_strings)

## Example 3 — `ensure_json` completes the text, and both fixers agree

In [ ]:
import json
from partial_json_parser import ensure_json, fix, fix_fast
from partial_json_parser.core.options import Allow

for text in ('{"a": [1, {"b": tru', '[null, "x", {"k": 1', '{"n": -1.5e'):
    completed = ensure_json(text, Allow.ALL)
    slow = "".join(fix(text, Allow.ALL))
    fast = "".join(fix_fast(text, Allow.ALL))
    assert json.loads(completed) == json.loads(slow) == json.loads(fast), (text, completed, slow, fast)
    print(repr(text), "->", completed)

## Example 4 — complete JSON round-trips unchanged

In [ ]:
import json
from partial_json_parser import loads

document = {"tools": [{"name": "f", "args": {"x": 1.5, "ok": True, "none": None, "items": [1, "two", [3]]}}]}
text = json.dumps(document)
assert loads(text) == document
# Every prefix of a valid document must parse to something, never crash with an unrelated error.
parsed_prefixes = sum(1 for cut in range(1, len(text)) if loads(text[:cut]) is not None)
assert parsed_prefixes == len(text) - 1, (parsed_prefixes, len(text))
print("full document round-trips;", parsed_prefixes, "prefixes parsed")

## Example 5 — the errors vLLM relies on

In [ ]:
import json
from partial_json_parser import loads
from partial_json_parser.core.exceptions import MalformedJSON, PartialJSON
from partial_json_parser.core.options import Allow

# A complete object followed by more text: vLLM's `partial_json_loads` reads the "Extra data" message.
try:
    loads('{"a": 1} trailing', Allow.ALL)
    raise AssertionError("extra data was accepted")
except json.JSONDecodeError as error:
    assert "Extra data" in str(error), str(error)
    print("extra data refused:", error)
# An unfinished object when partial objects are not allowed, and text that is not JSON at all.
for text, allow, expected in (('{"a": 1', Allow.STR, PartialJSON), ("wrong", Allow.ALL, MalformedJSON)):
    try:
        loads(text, allow)
        raise AssertionError(f"{text!r} was accepted")
    except expected as error:
        print(expected.__name__, "raised for", repr(text))